# GM6208-150T BEMF constant + shape — hand-spin capture

Extracts `MotorParams.ke_v_per_mech_rad_s`, verifies the pole-pair count, and
measures the BEMF shape against the model's ideal trapezoid, from a
bridge-disarmed hand-spin.

**Input:** `rotor_free_spin.csv` — `PCS_BENCH_DUTY_SEQ = 2` telemetry
(`motor_angle` + `phase_{u,v,w}_v` at 2 ms) logged by `tools/serial_capture.py`
while the rotor is spun by hand at varied speeds, both directions, coasting
between spins.

**Method:** ω from the unwrapped encoder angle. **Line-to-line** vsense
differences cancel the floating common mode; samples where any phase touches
the rails (diode clamping at speed) are masked. Shape: `e_uv/ω` binned over
electrical angle. Ke: windowed line-to-line peak against |ω| through the
origin; per-phase `Ke = slope/2` (the ideal trapezoid's flat-top overlap puts
the l-l peak at `2·Ke·ω`). Coast intervals give the friction ratios `B/J` and
`T_c/J` as a bonus. Run with the repo venv kernel (`.venv`).

In [ ]:
import csv
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

CSV = Path("rotor_free_spin.csv")
POLE_PAIRS = 14           # 24N28P per the product page; verified below
RAIL_LO_V, RAIL_HI_V = 0.15, 19.0   # clip mask: any phase outside -> sample invalid

raw = defaultdict(list)
with open(CSV) as f:
    for row in csv.DictReader(f):
        raw[row["signalName"].split("/")[-1]].append((int(row["t"]), float(row["value"])))
sig = {name: (np.array([t for t, _ in v]) * 1e-3, np.array([x for _, x in v]))
       for name, v in sorted(raw.items())}
t0 = min(t[0] for t, _ in sig.values())
n = min(len(v) for _, v in sig.values())
t = sig["motor_angle"][0][:n] - t0
ang_deg = sig["motor_angle"][1][:n]
V = np.stack([sig[f"phase_{p}_v"][1][:n] for p in ["u", "v", "w"]])
print(f"{n} samples over {t[-1] - t[0]:.1f} s")

## Mechanical state + validity mask

ω from the unwrapped encoder angle, lightly smoothed (30 ms boxcar) against
encoder quantization. A sample is *valid* when no phase touches a rail and
the speed is high enough for BEMF to clear the vsense resolution.

In [ ]:
theta = np.unwrap(np.deg2rad(ang_deg))
dt_s = float(np.median(np.diff(t)))
omega_raw = np.gradient(theta, t)
k = 15
kern = np.ones(k) / k
omega = np.convolve(omega_raw, kern, mode="same")

unclipped = np.all((V > RAIL_LO_V) & (V < RAIL_HI_V), axis=0)
W_MIN = 4.0
valid = unclipped & (np.abs(omega) > W_MIN)
print(f"dt = {dt_s * 1e3:.1f} ms; omega {omega.min():.1f}..{omega.max():.1f} rad/s")
print(f"valid samples: {valid.sum()} of {n} "
      f"({unclipped.sum()} unclipped, |omega| > {W_MIN})")

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
axes[0].plot(t, omega, lw=0.7)
axes[0].set_ylabel("omega [rad/s]")
for kph, p in enumerate(["u", "v", "w"]):
    axes[1].plot(t, V[kph], lw=0.3, label=f"v_{p}")
axes[1].set_ylabel("vsense [V]")
axes[1].set_xlabel("t [s]")
axes[1].legend(ncol=3)
fig.suptitle("hand-spin capture: speed + terminal voltages")
fig.tight_layout()

## Pole-pair check

Electrical cycles per mechanical revolution: zero crossings of `e_uv` per rev
over the valid stretches — 2 crossings per electrical cycle, so
`pole pairs = crossings / (2·revs)`.

In [ ]:
e_uv = V[0] - V[1]
runs = []
start = None
for i, ok in enumerate(valid):
    if ok and start is None:
        start = i
    if (not ok or i == n - 1) and start is not None:
        if i - start > 500:   # >= 1 s of continuous valid spin
            runs.append((start, i))
        start = None

total_cross, total_revs = 0, 0.0
for s, e in runs:
    x = e_uv[s:e] - np.mean(e_uv[s:e])
    total_cross += int(np.sum(np.abs(np.diff(np.sign(x))) == 2))
    total_revs += abs(theta[e] - theta[s]) / (2 * np.pi)
pp_meas = total_cross / (2.0 * total_revs)
print(f"{len(runs)} valid spin runs, {total_revs:.1f} mech revs, "
      f"{total_cross} e_uv zero crossings")
print(f"pole pairs = {pp_meas:.2f}   (config: {POLE_PAIRS})")

## BEMF shape: `e_uv / ω` binned over electrical angle

Normalizing by ω collapses every speed onto one curve — its amplitude is the
line-to-line Ke, its form is the (line-to-line) BEMF shape. The ideal-model
overlay is `f(θe+φ) − f(θe+φ−2π/3)` with φ grid-searched (the encoder's
electrical phase is arbitrary). Line-to-line hides triplen (3rd-harmonic)
content by construction — which is also all the drive can ever see with an
isolated neutral, so this is the shape that matters for torque.

In [ ]:
def bemf_shape(th):
    tt = np.mod(th, 2 * np.pi)
    return np.select(
        [tt < np.pi / 6, tt < 5 * np.pi / 6, tt < 7 * np.pi / 6, tt < 11 * np.pi / 6],
        [(6 / np.pi) * tt, 1.0, (6 / np.pi) * (np.pi - tt), -1.0],
        default=(6 / np.pi) * (tt - 2 * np.pi))

theta_e = np.mod(theta * POLE_PAIRS, 2 * np.pi)
g = e_uv[valid] / omega[valid]
te = theta_e[valid]

NBINS = 120
bins = np.linspace(0, 2 * np.pi, NBINS + 1)
centers = 0.5 * (bins[:-1] + bins[1:])
idx = np.digitize(te, bins) - 1
binned = np.array([np.median(g[idx == b]) if np.any(idx == b) else np.nan
                   for b in range(NBINS)])
counts = np.array([np.sum(idx == b) for b in range(NBINS)])
ke_ll = float(np.nanmax(np.abs(binned)))

# Candidate line-to-line shapes, each normalized to unit peak so the overlay
# carries the measured amplitude: the model's trapezoid (l-l peak 2) and a pure
# sinusoid. Phase, sign, and phase sequence are grid-searched per candidate.
def ll_shape(kind, th, seq):
    if kind == "trapezoid":
        return (bemf_shape(th) - bemf_shape(th - seq * 2 * np.pi / 3)) / 2.0
    return np.sin(th) - np.sin(th - seq * 2 * np.pi / 3)  # peak sqrt(3)...


def ll_unit(kind, th, seq):
    s = ll_shape(kind, th, seq)
    return s / (2.0 / 2.0 if kind == "trapezoid" else np.sqrt(3.0))


phis = np.linspace(0, 2 * np.pi, 720, endpoint=False)
results = {}
for kind in ["trapezoid", "sinusoid"]:
    best = max(((np.nansum(binned * ll_unit(kind, centers + phi, seq)), phi, seq)
                for phi in phis for seq in (+1.0, -1.0)), key=lambda q: abs(q[0]))
    ideal = np.sign(best[0]) * ke_ll * ll_unit(kind, centers + best[1], best[2])
    rms = float(np.sqrt(np.nanmean((binned - ideal) ** 2))) / ke_ll
    results[kind] = dict(ideal=ideal, rms=rms, seq=best[2])

winner = min(results, key=lambda k: results[k]["rms"])
seq_name = "U->V->W" if results[winner]["seq"] > 0 else "U->W->V"

# Harmonic content of the binned curve (cycles per electrical period).
spec = 2 * np.abs(np.fft.rfft(np.nan_to_num(binned - np.nanmean(binned))) / NBINS)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(te, g, ".", ms=1, alpha=0.15, label="samples e_uv/omega")
ax.plot(centers, binned, "b-", lw=1.8, label="binned median")
ax.plot(centers, results["sinusoid"]["ideal"], "r--", lw=1.4,
        label=f"sinusoid ({results['sinusoid']['rms'] * 100:.1f}% rms)")
ax.plot(centers, results["trapezoid"]["ideal"], "g:", lw=1.4,
        label=f"trapezoid ({results['trapezoid']['rms'] * 100:.1f}% rms)")
ax.set_xlabel("electrical angle [rad]")
ax.set_ylabel("e_uv / omega [V·s/rad]")
ax.legend()
ax.set_title(f"line-to-line BEMF shape — best fit: {winner}")
fig.tight_layout()

print(f"binned-shape amplitude (Ke line-to-line) : {ke_ll:.4f} V·s/rad")
for kind, r in results.items():
    print(f"rms deviation, {kind:9s}                : {r['rms'] * 100:5.1f}% of peak")
print(f"winning shape                            : {winner}")
print(f"phase sequence vs encoder-positive       : {seq_name}")
print("harmonics h1..h7: " + "  ".join(f"{spec[h]:.4f}" for h in range(1, 8)))
print(f"thinnest bin: {counts.min()} samples")

## Ke linearity: windowed l-l peak against |ω|

Every 100 ms window that is fully valid contributes (peak |e_uv|, mean |ω|).
A through-origin fit across all speeds gives the line-to-line peak constant;
the model's per-phase `ke_v_per_mech_rad_s` is half of it.

In [ ]:
WIN = 50   # 100 ms
pk, wm = [], []
for s in range(0, n - WIN, WIN):
    sl = slice(s, s + WIN)
    if np.all(valid[sl]):
        pk.append(np.max(np.abs(e_uv[sl])))
        wm.append(np.mean(np.abs(omega[sl])))
pk, wm = np.array(pk), np.array(wm)
ke_ll_fit = float(np.sum(pk * wm) / np.sum(wm * wm))
resid = pk - ke_ll_fit * wm

# Per-phase conversion follows the winning shape: the l-l peak is 2·Ke for the
# trapezoid's flat-top overlap, sqrt(3)·Ke for a sinusoid.
ll_factor = 2.0 if winner == "trapezoid" else float(np.sqrt(3.0))
ke_phase = ke_ll_fit / ll_factor

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(wm, pk, ".", ms=4, alpha=0.5, label=f"{len(pk)} windows")
xs = np.linspace(0, wm.max() * 1.05, 50)
ax.plot(xs, ke_ll_fit * xs, "r-", lw=1.2, label=f"fit: {ke_ll_fit:.4f} V·s/rad")
ax.set_xlabel("|omega| [rad/s]")
ax.set_ylabel("peak |e_uv| [V]")
ax.legend()
ax.set_title("BEMF is proportional to speed")
fig.tight_layout()

print("=" * 62)
print("GM6208-150T BEMF constant — hand-spin fit")
print("=" * 62)
print(f"Ke line-to-line peak (V-I fit)   : {ke_ll_fit:.4f} V·s/rad")
print(f"Ke line-to-line peak (shape amp) : {ke_ll:.4f} V·s/rad")
print(f"fit residual rms                 : {np.sqrt(np.mean(resid ** 2)):.3f} V")
print(f"winning shape ({winner:9s})       : l-l peak = {ll_factor:.3f} · Ke_phase")
print(f"model ke_v_per_mech_rad_s        : {ke_phase:.4f}   <- MotorParams")
print("=" * 62)

## Coast-down friction ratios (bonus)

Hands-off decel stretches obey `ω̇ = −(B/J)·ω − (T_c/J)·sign(ω)`: a linear
fit of ω̇ against ω over the coasts separates viscous slope from the Coulomb
intercept. These are *ratios* — J itself comes from the alignment-ring
measurement.

In [ ]:
kk = 51
om_s = np.convolve(omega_raw, np.ones(kk) / kk, mode="same")
al_s = np.gradient(om_s, t)
# Coast candidates: spinning, decelerating in magnitude, away from clip.
coast = (np.abs(om_s) > 2.0) & (np.abs(om_s) < 25.0) & (al_s * np.sign(om_s) < 0)
wpts, apts = om_s[coast], al_s[coast]
A = np.stack([wpts, np.sign(wpts)], axis=1)
(bj, tcj), *_ = np.linalg.lstsq(A, -apts, rcond=None)
print(f"coast samples: {coast.sum()}")
print(f"B/J  = {bj:.3f} 1/s      (viscous; tau_mech = J/B = {1 / bj if bj > 0 else float('nan'):.2f} s)")
print(f"Tc/J = {tcj:.3f} rad/s^2 (Coulomb)")
print("absolute B and T_c follow once J is measured (alignment ring-down)")

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(wpts, -apts, ".", ms=2, alpha=0.2)
xs = np.linspace(wpts.min(), wpts.max(), 100)
ax.plot(xs, bj * xs + tcj * np.sign(xs), "r-", lw=1.2,
        label=f"-d(omega)/dt = {bj:.3f}·omega + {tcj:.2f}·sign")
ax.set_xlabel("omega [rad/s]")
ax.set_ylabel("-d(omega)/dt [rad/s²]")
ax.legend()
ax.set_title("coast-down deceleration vs speed")
fig.tight_layout()

## Caveats

- Samples with any phase at a rail are masked: fast flicks push BEMF past the
  bus window and the body diodes clamp (real rectification — the same physics
  the SIL overspeed test exercises). Diode conduction during those stretches
  also loads the rotor, so coast fits exclude them via the |ω| < 25 cut.
- Line-to-line analysis cannot see triplen (3rd-harmonic) BEMF content — and
  neither can any drive across an isolated neutral, so the fitted shape is
  the torque-relevant one. The model's per-phase `bemf_shape` maps to it via
  `f(θ) − f(θ − 2π/3)`.
- ω derives from the noisy encoder (σ ≈ 1.5 LSB): the 30 ms smoothing biases
  nothing at hand-spin bandwidths but caps the usable acceleration detail.
- Hand-spin torque is unknown, so only torque-free (coast) stretches inform
  the friction fit; the spin-up strokes are excluded by the deceleration
  condition.